# Soil Physicochemical Properties — Glenlea Long-Term Crop Rotation

**M.Sc. Research Project**  
**Department of Soil Science, University of Manitoba**

**Project:** *Shotgun Metagenomic Assessment of Microbial Communities and Pesticide Impacts in Biomixtures and Agricultural Soils*

---

## Research context

This notebook documents the reproducible analysis of soil physicochemical properties measured in **15 soil samples** collected from the Glenlea Long-Term Crop Rotation.

The sampled management/rotation groups are:

| Group | n |
|---|---:|
| 1-Organic | 3 |
| 16-Organic | 3 |
| 1-Conventional | 2 |
| 16-Conventional | 4 |
| Native grass | 3 |
| **Total** | **15** |

The current analysis is **descriptive**. Inferential statistical testing will be added after the statistical analysis plan is finalized.


## Analytical variables

The workbook contains measurements for:

- Organic Matter (OM, %)
- Olsen Phosphorus (P-Olsen, ppm)
- Total Nitrogen (Total-N, %)
- Total Carbon (T.C., %)
- Calcium Carbonate Equivalent (CCE, %)
- Total Organic Carbon (T.O.C., %)
- Soil pH
- Electrical Conductivity (EC, dS/m)

The visualizations in this notebook focus on the six soil properties currently presented in the project README: **OM, Olsen P, TOC, CCE, pH, and EC**.


## 1. Analysis setup


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

DATA_FILE = Path("../data/glenlea-soil-properties.xlsx")
SHEET_NAME = "UN8619"


## 2. Import and prepare the dataset

The first row of the worksheet contains variable names and the second row contains units.  
The units row is removed during import, and field identifiers are standardized before analysis.


In [ ]:
raw = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)

# Remove the units row and standardize column names.
df = raw.iloc[1:].copy()
df.columns = [str(col).strip() for col in df.columns]

# Standardize field identifiers and depth values.
df["Field ID"] = df["Field ID"].astype(str).str.strip()
df["Depth"] = df["Depth"].astype(str).str.strip()

numeric_columns = [
    "Sample ID", "OM", "P-Olsen", "Total-N", "T.C.",
    "CCE", "T.O.C.", "pH", "EC"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.reset_index(drop=True, inplace=True)

df.head()


## 3. Assign management/rotation groups

Group labels are generated directly from the Glenlea field identifiers so that the grouping step is reproducible.


In [ ]:
def assign_group(field_id):
    field_id = field_id.upper().strip()

    if "NATIVE GRASS" in field_id:
        return "Native grass"
    if field_id.startswith("1-O-"):
        return "1-Organic"
    if field_id.startswith("16-O-"):
        return "16-Organic"
    if field_id.startswith("1-C-"):
        return "1-Conventional"
    if field_id.startswith("16-C-"):
        return "16-Conventional"

    return "Unclassified"

df["Group"] = df["Field ID"].apply(assign_group)

group_order = [
    "1-Organic",
    "16-Organic",
    "1-Conventional",
    "16-Conventional",
    "Native grass",
]

df["Group"] = pd.Categorical(
    df["Group"],
    categories=group_order,
    ordered=True
)

df[["Field ID", "Sample ID", "Depth", "Group"]]


## 4. Data quality and sample verification

The imported dataset is checked for:

- expected sample number (`n = 15`)
- missing values
- correct treatment assignment
- numeric data types

These checks document the condition of the dataset before analysis.


In [ ]:
print(f"Number of soil samples: {len(df)}")

sample_counts = (
    df.groupby("Group", observed=False)
      .size()
      .reindex(group_order)
      .rename("n")
)

display(sample_counts.to_frame())

missing_values = df[
    ["OM", "P-Olsen", "Total-N", "T.C.", "CCE", "T.O.C.", "pH", "EC"]
].isna().sum().rename("Missing values")

display(missing_values.to_frame())


## 5. Descriptive statistics

For each management/rotation group, descriptive statistics are calculated for the measured soil properties.

The table reports:

- sample number (`n`)
- mean
- standard deviation
- median
- minimum
- maximum

These summaries describe the observed data and do not by themselves indicate statistically significant differences among groups.


In [ ]:
analysis_variables = [
    "OM", "P-Olsen", "Total-N", "T.C.",
    "CCE", "T.O.C.", "pH", "EC"
]

descriptive_statistics = (
    df.groupby("Group", observed=False)[analysis_variables]
      .agg(["count", "mean", "std", "median", "min", "max"])
      .round(3)
)

descriptive_statistics


## 6. Soil-property visualizations

Box plots are used to show the distribution of observations within each management/rotation group.  
Individual sample observations are displayed because group sizes are small (`n = 2–4`).


In [ ]:
def plot_soil_property(data, variable, title, y_label):
    grouped_values = []
    labels = []

    for group in group_order:
        values = data.loc[data["Group"] == group, variable].dropna()
        grouped_values.append(values.to_numpy())
        labels.append(f"{group}\n(n={len(values)})")

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.boxplot(
        grouped_values,
        labels=labels,
        showmeans=True
    )

    # Overlay individual observations with small deterministic horizontal offsets.
    for position, values in enumerate(grouped_values, start=1):
        if len(values) == 1:
            x_positions = [position]
        else:
            span = 0.10
            step = (2 * span) / (len(values) - 1)
            x_positions = [position - span + i * step for i in range(len(values))]
        ax.scatter(x_positions, values, zorder=3)

    ax.set_title(title)
    ax.set_ylabel(y_label)
    ax.set_xlabel("")
    fig.tight_layout()
    plt.show()


### Total Organic Carbon

In [ ]:
plot_soil_property(
    df,
    variable="T.O.C.",
    title="Total Organic Carbon",
    y_label="Total Organic Carbon (%)"
)


### Organic Matter

In [ ]:
plot_soil_property(
    df,
    variable="OM",
    title="Organic Matter",
    y_label="Organic Matter (%)"
)


### Soil pH

In [ ]:
plot_soil_property(
    df,
    variable="pH",
    title="Soil pH",
    y_label="Soil pH"
)


### Electrical Conductivity

In [ ]:
plot_soil_property(
    df,
    variable="EC",
    title="Electrical Conductivity",
    y_label="Electrical Conductivity (dS/m)"
)


### Calcium Carbonate Equivalent

In [ ]:
plot_soil_property(
    df,
    variable="CCE",
    title="Calcium Carbonate Equivalent",
    y_label="Calcium Carbonate Equivalent (%)"
)


### Olsen Phosphorus

In [ ]:
plot_soil_property(
    df,
    variable="P-Olsen",
    title="Olsen Phosphorus",
    y_label="Olsen Phosphorus (ppm)"
)


## 7. Additional measured soil properties

Total nitrogen and total carbon are retained in the analysis dataset even though they are not currently displayed in the main project figure set.

They can be incorporated into future statistical analyses or additional visualizations as the thesis analysis develops.


In [ ]:
additional_properties = (
    df.groupby("Group", observed=False)[["Total-N", "T.C."]]
      .agg(["count", "mean", "std", "median", "min", "max"])
      .round(3)
)

additional_properties


## 8. Statistical analysis status

Inferential statistical analysis has **not yet been completed**.

The final analysis will be selected based on the experimental structure and statistical assumptions. Planned work includes:

- evaluation of distributional assumptions and variance structure
- appropriate treatment/management comparisons
- analysis of variance (ANOVA) where justified
- post-hoc comparisons where appropriate
- integration of soil-property data with microbial-community and pesticide-residue datasets

Statistical analyses are planned in **SAS**.


## 9. Interpretation

At this stage, the figures and summary tables are used to describe the **observed patterns** among Glenlea management/rotation groups.

Because formal inferential testing has not yet been completed, the notebook intentionally avoids statements of statistical significance.

Scientific interpretation will be expanded after the statistical analyses are completed and will then be integrated with:

- shotgun metagenomic community profiles
- microbial diversity metrics
- pesticide residue concentrations


## 10. Reproducibility and next steps

- [x] Import AGVISE soil-property dataset
- [x] Standardize sample identifiers
- [x] Assign Glenlea management/rotation groups
- [x] Verify sample counts and missing values
- [x] Calculate descriptive statistics
- [x] Generate preliminary soil-property box plots
- [ ] Complete inferential statistical analysis
- [ ] Add statistical annotations to final figures where appropriate
- [ ] Integrate soil properties with microbial-community data
- [ ] Integrate soil properties with pesticide-residue data

---

**Research status:** Ongoing M.Sc. research. Results in this notebook should be considered preliminary unless explicitly identified as final.
